# RAG Workflow with LangGraph

Now that we can index documents (see `indexing.ipynb`), let's actually answer questions with them. A **RAG workflow** chains two steps:

```
user question  →  [retrieve]  →  [generate]  →  answer
```

- **retrieve**: find the chunks most relevant to the question.
- **generate**: ask the LLM to answer the question using those chunks as context.

We'll wire this up as a tiny `StateGraph` — exactly the same LangGraph pattern you saw in the dynamic-context notebook.

## Setup

Load environment variables from your `.env` file. The Azure endpoint is pasted directly on the embeddings model below.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, AzureOpenAIEmbeddings

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint="",  # TODO: paste your Azure endpoint here
)

print("Setup complete.")

## Step 1: Build the index

Quick recap from the indexing notebook: load the PDF, split it, and store the chunks in a vector store.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

loader = PyPDFLoader(
    "https://www.vestas.com/content/dam/vestas-com/global/en/investor/reports-and-presentations/financial/2024/fy-2024/Vestas%20Annual%20Report%202024.pdf.coredownload.inline.pdf"
)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = splitter.split_documents(docs)

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(all_splits)

print(f"Indexed {len(all_splits)} chunks.")

## Step 2: Define the workflow state

The workflow needs to carry two things between nodes:

- `messages`: the conversation
- `context`: the chunks the retriever found

We'll define this as a `TypedDict` (just like in the dynamic-context notebook).

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph.message import add_messages


class RAGState(TypedDict):
    messages: Annotated[list, add_messages]  # `add_messages` reducer appends new messages
    context: list[Document]

## Step 3: The `retrieve` node

Reads the latest user message, queries the vector store, and stuffs the result into `context`.

In [ ]:
def retrieve(state: RAGState) -> dict:
    query = state["messages"][-1].content
    retrieved_docs = vector_store.similarity_search(query, k=4)
    print(f"Retrieved {len(retrieved_docs)} chunks for: '{query}'")
    return {"context": retrieved_docs}

## Step 4: The `generate` node

Reads `context`, formats it into a system prompt, and asks the LLM.

In [ ]:
def generate(state: RAGState) -> dict:
    question = state["messages"][-1].content
    context_text = "\n\n---\n\n".join(
        f"[Page {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for doc in state["context"]
    )

    system_prompt = (
        "You are a helpful assistant answering questions about the Vestas Annual Report 2024. "
        "Use the context below. If the answer isn't in the context, say so.\n\n"
        f"## Context:\n\n{context_text}"
    )

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ])

    return {"messages": [response]}

## Step 5: Wire up the graph

`START → retrieve → generate → END`

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

rag_workflow = builder.compile()
rag_workflow

## Step 6: Run it!

In [ ]:
result = rag_workflow.invoke({
    "messages": [{"role": "user", "content": "What was Vestas' total revenue in 2024?"}],
    "context": [],
})

for message in result["messages"]:
    message.pretty_print()

In [ ]:
result = rag_workflow.invoke({
    "messages": [{"role": "user", "content": "How many employees does Vestas have?"}],
    "context": [],
})

for message in result["messages"]:
    message.pretty_print()

## Try it yourself 🛠️

Make the assistant **cite the page numbers** it used.

1. Update the `system_prompt` inside `generate` to instruct the model: *"After your answer, list the page numbers you used (e.g. `Sources: pages 12, 47`)."*
2. Rebuild the workflow (re-run the graph cell) and re-run a query.
3. Check the response — does it list pages now?

**Hint:** the page numbers are already in the prompt as `[Page X]` markers — you just need to tell the model to surface them.

In [ ]:
# Your solution here

def generate(state: RAGState) -> dict:
    question = state["messages"][-1].content
    context_text = "\n\n---\n\n".join(
        f"[Page {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for doc in state["context"]
    )

    # TODO: extend the system prompt so the model lists the source page numbers.
    system_prompt = (
        "You are a helpful assistant answering questions about the Vestas Annual Report 2024. "
        "Use the context below. If the answer isn't in the context, say so.\n\n"
        f"## Context:\n\n{context_text}"
    )

    response = llm.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ])
    return {"messages": [response]}


# Rebuild the workflow with the updated generate node
builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
rag_workflow = builder.compile()

result = rag_workflow.invoke({
    "messages": [{"role": "user", "content": "What are Vestas' main sustainability initiatives?"}],
    "context": [],
})

for message in result["messages"]:
    message.pretty_print()